In [11]:
#Importação das bibliotecas
import sys
import glob
import os
import pandas as pd

sys.path.append("..")
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, get_logger

logger = get_logger("01_extracao_e_limpeza")
logger.info("Iniciando pipeline de extração e limpeza...")

20:58:36 | INFO     | 01_extracao_e_limpeza | Iniciando pipeline de extração e limpeza...


In [12]:
# Localizar todos os arquivos CSV na pasta data/raw
csv_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "bdqueimadas_*.csv")))

if not csv_files:
    raise FileNotFoundError(
        f"Nenhum arquivo 'bdqueimadas_*.csv' encontrado em {RAW_DATA_DIR}. "
        "Verifique se os arquivos brutos foram colocados na pasta correta."
    )

logger.info(f"Encontrados {len(csv_files)} arquivos para processar:")
for f in csv_files:
    logger.info(f"  - {os.path.basename(f)}")


20:58:36 | INFO     | 01_extracao_e_limpeza | Encontrados 6 arquivos para processar:
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2021-01-01_2022-01-01.csv
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2022-01-02_2023-01-01.csv
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2023-01-02_2024-01-01.csv
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2024-01-02_2025-01-01.csv
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2025-01-02_2026-01-01.csv
20:58:36 | INFO     | 01_extracao_e_limpeza |   - bdqueimadas_2026-01-02_2026-07-16.csv


In [13]:
# Ler e concatenar todos os arquivos em um único DataFrame,
# validando que todos compartilham o mesmo schema antes de concatenar.
df_list = []
schema_referencia = None

for file in csv_files:
    try:
        temp_df = pd.read_csv(file, encoding="utf-8")
    except UnicodeDecodeError:
        logger.warning(f"{os.path.basename(file)}: encoding utf-8 falhou, tentando iso-8859-1.")
        temp_df = pd.read_csv(file, encoding="iso-8859-1")
    except Exception as e:
        logger.error(f"Falha ao ler {os.path.basename(file)}: {e}")
        raise

    colunas_atuais = set(temp_df.columns)
    if schema_referencia is None:
        schema_referencia = colunas_atuais
    elif colunas_atuais != schema_referencia:
        faltando = schema_referencia - colunas_atuais
        extras = colunas_atuais - schema_referencia
        logger.warning(
            f"{os.path.basename(file)}: schema divergente do arquivo de referência. "
            f"Faltando: {faltando or 'nenhuma'} | Extras: {extras or 'nenhuma'}"
        )
        # Não interrompe o pipeline, mas avisa explicitamente — colunas ausentes
        # viram NaN após o concat, o que precisa ser uma decisão consciente, não silenciosa.

    temp_df["source_file"] = os.path.basename(file)
    df_list.append(temp_df)

df_raw = pd.concat(df_list, ignore_index=True)

logger.info("Concatenação concluída.")
logger.info(f"Total de registros importados: {df_raw.shape[0]:,}")
logger.info(f"Total de colunas originais: {df_raw.shape[1]}")


20:58:37 | INFO     | 01_extracao_e_limpeza | Concatenação concluída.
20:58:37 | INFO     | 01_extracao_e_limpeza | Total de registros importados: 1,011,372
20:58:37 | INFO     | 01_extracao_e_limpeza | Total de colunas originais: 13


In [14]:
# Processamento, Limpeza e Otimização
logger.info("Iniciando tratamento de dados...")
df_clean = df_raw.copy()  # cópia para não alterar o df_raw original

# Padronização do nome das colunas
colunas_novas = {
    "DataHora": "data_hora",
    "Satelite": "satelite",
    "Pais": "pais",
    "Estado": "estado",
    "Municipio": "municipio",
    "Bioma": "bioma",
    "DiaSemChuva": "dias_sem_chuva",
    "Precipitacao": "precipitacao",
    "RiscoFogo": "risco_fogo",
    "FRP": "frp",
    "Latitude": "latitude",
    "Longitude": "longitude",
}
df_clean = df_clean.rename(columns=colunas_novas)

# Convertendo a data de texto para datetime
df_clean["data_hora"] = pd.to_datetime(df_clean["data_hora"], format="%Y/%m/%d %H:%M:%S", errors="coerce")

n_datas_invalidas = df_clean["data_hora"].isna().sum()
if n_datas_invalidas > 0:
    logger.warning(f"{n_datas_invalidas:,} registros com data_hora inválida (viraram NaT).")

df_clean["ano"] = df_clean["data_hora"].dt.year
df_clean["mes"] = df_clean["data_hora"].dt.month
df_clean["dia"] = df_clean["data_hora"].dt.day


20:58:37 | INFO     | 01_extracao_e_limpeza | Iniciando tratamento de dados...


In [15]:
# Tratamento de valores nulos em indicadores meteorológicos.
for col in ["dias_sem_chuva", "precipitacao", "risco_fogo"]:
    n_negativos = (df_clean[col] < 0).sum()
    if n_negativos > 0: 
        logger.warning( #
            f"Coluna '{col}': {n_negativos:,} valores negativos encontrados "
            "(possível valor sentinela de sensor sem leitura, ex: -999). "
            "Revise antes de assumir que são dados válidos."
        )

    n_nulos = df_clean[col].isna().sum()
    if n_nulos > 0:
        mediana = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(mediana)
        logger.info(f"Coluna '{col}': {n_nulos:,} nulos preenchidos com a mediana ({mediana:.2f}).")


20:58:38 | WARNING  | 01_extracao_e_limpeza | Coluna 'dias_sem_chuva': 14,733 valores negativos encontrados (possível valor sentinela de sensor sem leitura, ex: -999). Revise antes de assumir que são dados válidos.
20:58:38 | INFO     | 01_extracao_e_limpeza | Coluna 'dias_sem_chuva': 231 nulos preenchidos com a mediana (8.00).
20:58:38 | INFO     | 01_extracao_e_limpeza | Coluna 'precipitacao': 231 nulos preenchidos com a mediana (0.00).
20:58:38 | WARNING  | 01_extracao_e_limpeza | Coluna 'risco_fogo': 6,500 valores negativos encontrados (possível valor sentinela de sensor sem leitura, ex: -999). Revise antes de assumir que são dados válidos.
20:58:38 | INFO     | 01_extracao_e_limpeza | Coluna 'risco_fogo': 231 nulos preenchidos com a mediana (0.95).


In [16]:
# Padronização de textos.
colunas_texto = ["estado", "municipio", "bioma", "satelite", "pais"]
for col in colunas_texto:
    n_nulos_antes = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].str.strip().str.upper()
    if n_nulos_antes > 0:
        logger.info(f"Coluna '{col}': {n_nulos_antes:,} valores nulos preservados como NULL (não convertidos para texto).")


In [17]:
# Salvando os dados processados em formato otimizado Parquet, com fallback para CSV.
caminho_saida_parquet = os.path.join(PROCESSED_DATA_DIR, "queimadas_limpo.parquet")
caminho_saida_csv = os.path.join(PROCESSED_DATA_DIR, "queimadas_limpo.csv")

try:
    df_clean.to_parquet(caminho_saida_parquet, index=False)
    logger.info(f"Dados salvos com sucesso em formato Parquet: {caminho_saida_parquet}")
    logger.info(
        "Se este é o formato principal do projeto, confirme que o notebook 02 "
        "está lendo o .parquet (não o .csv) para evitar dados desatualizados."
    )
except Exception as e:
    logger.error(f"Erro ao salvar em Parquet ({e}). Verifique se 'pyarrow' está instalado. Salvando em CSV de backup...")
    df_clean.to_csv(caminho_saida_csv, index=False)
    logger.info(f"Dados salvos em CSV processado: {caminho_saida_csv}")


20:58:38 | INFO     | 01_extracao_e_limpeza | Dados salvos com sucesso em formato Parquet: /home/eric/projetos/Analise-Queimadas-Brasil/data/processed/queimadas_limpo.parquet
20:58:38 | INFO     | 01_extracao_e_limpeza | Se este é o formato principal do projeto, confirme que o notebook 02 está lendo o .parquet (não o .csv) para evitar dados desatualizados.


In [18]:
logger.info("Informações finais do DataFrame processado:")
df_clean.info()

20:58:38 | INFO     | 01_extracao_e_limpeza | Informações finais do DataFrame processado:


<class 'pandas.DataFrame'>
RangeIndex: 1011372 entries, 0 to 1011371
Data columns (total 16 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   data_hora       1011372 non-null  datetime64[us]
 1   satelite        1011372 non-null  str           
 2   pais            1011372 non-null  str           
 3   estado          1011372 non-null  str           
 4   municipio       1011372 non-null  str           
 5   bioma           1011372 non-null  str           
 6   dias_sem_chuva  1011372 non-null  float64       
 7   precipitacao    1011372 non-null  float64       
 8   risco_fogo      1011372 non-null  float64       
 9   frp             1011372 non-null  float64       
 10  latitude        1011372 non-null  float64       
 11  longitude       1011372 non-null  float64       
 12  source_file     1011372 non-null  str           
 13  ano             1011372 non-null  int32         
 14  mes             1011372 non-n

In [19]:
# Checagem de integridade: existe algum mês sem nenhum registro dentro do range de dados
meses_com_dado = df_clean["data_hora"].dt.to_period("M").unique()
todos_os_meses = pd.period_range(df_clean["data_hora"].min(), df_clean["data_hora"].max(), freq="M")
meses_faltando = set(todos_os_meses) - set(meses_com_dado)
print(sorted(meses_faltando))

[]


In [26]:
contagem_mensal = df_clean.groupby(df_clean["data_hora"].dt.to_period("M")).size()
print(contagem_mensal.to_string())

data_hora
2021-01     2237
2021-02     2187
2021-03     2501
2021-04     2548
2021-05     5288
2021-06     7470
2021-07    15985
2021-08    51711
2021-09    49829
2021-10    28342
2021-11    11596
2021-12     4387
2022-01     2759
2022-02     1931
2022-03     1790
2022-04     1616
2022-05     6698
2022-06     7876
2022-07    14212
2022-08    47507
2022-09    60313
2022-10    32130
2022-11    16978
2022-12     6953
2023-01     2494
2023-02     2035
2023-03     2585
2023-04     2359
2023-05     5286
2023-06     8597
2023-07    13985
2023-08    28054
2023-09    46486
2023-10    39692
2023-11    26819
2023-12    11509
2024-01     4555
2024-02     4566
2024-03     5448
2024-04     2613
2024-05     6324
2024-06    12432
2024-07    22478
2024-08    68635
2024-09    83154
2024-10    33356
2024-11    22663
2024-12    12075
2025-01     3137
2025-02     1819
2025-03     2357
2025-04     1641
2025-05     4263
2025-06     6060
2025-07     9803
2025-08    18451
2025-09    28214
2025-10    28678
2025